# Compact MI LOSO Transfer
Target-excluded supervised source pretraining with target-fold adaptation.

# 1. Setup

In [ ]:
from __future__ import annotations
import builtins, hashlib, json, os, platform, random, sys, time
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
from sklearn.metrics import confusion_matrix
from modern_mi_common import *
print(f"Python: {sys.version.split()[0]} | Platform: {platform.platform()} | CWD: {Path.cwd()}")

# 2. Configuration
## 2.1 LOSO Defaults
## 2.2 CONFIG

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent
CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-compact-mi-loso-transfer"), "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"), "experiment_name": "compact_mi_loso_fbcnet_pilot", "config_note": "LOSO source pretraining and target adaptation pilot.",
    # Dataset and trial-independent preprocessing
    "subjects_to_use": [1, 3, 7, 9], "target_subjects": [1, 3], "channel_set": "liu29", "native_sfreq": 500, "target_sfreq": 128, "marker_channel_index": 32, "onset_marker_value": 2, "onset_plausible_range": [800,1300], "window_seconds": 4.0, "bandpass_hz": [4.0,40.0], "filter_order": 4, "normalization_mode": "channel_standardize", "normalization_eps": 1e-6,
    # Model / evaluation
    "model_name": "FBCNet", "model_kwargs": {}, "allow_optional_models": False, "adaptation_mode": "frozen_encoder", "source_epochs": 1, "adaptation_epochs": 5, "finetune_learning_rate": 0.00003, "checkpoint_cache_dir": str(WORKING_DIR / "artifacts" / "checkpoint_cache" / "compact-mi-loso"), "evaluation_mode": "stratified_5fold", "cv_folds": 5, "cv_random_state": 2026, "persist_splits": True, "epoch_selection": "fixed_source_locked",
    # Training / reproducibility / diagnostics
    "source_batch_size": 64, "batch_size": 8, "n_epochs": 20, "learning_rate": 0.0003, "weight_decay": 0.01, "gradient_clip_norm": 1.0, "seeds": [2026], "seed": 2026, "set_seed": True, "bootstrap_iterations": 10000, "collapse_threshold": 0.9, "classical_reference_artifact": str(WORKING_DIR / "artifacts" / "liu2024-multiscale-riemann-fusion" / "20260712_145032_844360_89d4f1fc" / "subject_results.csv")
}

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"
RUN_ID = create_run_id(); ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = ARTIFACT_DIR / "run.log"; _LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")
def _safe_write_text(stream, text):
    try: stream.write(text)
    except UnicodeEncodeError:
        enc = getattr(stream, "encoding", None) or "utf-8"; stream.write(text.encode(enc, errors="replace").decode(enc, errors="replace"))
def _timestamped_print(*args, **kwargs):
    sep=kwargs.pop("sep"," "); end=kwargs.pop("end","\n"); flush=kwargs.pop("flush",False); file=kwargs.pop("file",None)
    message=sep.join(str(a) for a in args); target=sys.stdout if file is None else file; stamped=f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {message}"
    _safe_write_text(target, stamped+end); _safe_write_text(_LOG_FILE_HANDLE, stamped+end)
    if flush: target.flush(); _LOG_FILE_HANDLE.flush()
builtins.print = _timestamped_print
config_path=ARTIFACT_DIR/"config.json"; config_path.write_text(json.dumps(CONFIG,indent=2),encoding="utf-8")
print(f"Run ID:     {RUN_ID}"); print(f"Artifacts:  {ARTIFACT_DIR}"); print(f"Config:     {config_path}")

## 2.4 Reproducibility

In [ ]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built(): return torch.device("mps")
    if torch.cuda.is_available(): return torch.device("cuda")
    return torch.device("cpu")
DEVICE=resolve_device(); print(f"Using device: {DEVICE}")
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"]=str(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed); torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True,warn_only=True)
BASE_SEED=int(CONFIG["seed"])
if CONFIG["set_seed"]: seed_everything(BASE_SEED); print(f"Seed initialized: {BASE_SEED}")

# 3. Load and Prepare Data
## 3.1 Data Loading Helpers
## 3.2 Fold-Safe Target Normalization
## 3.3 Dataset Classes
## 3.4 Locate and Load Data

# 4. Model
## 4.1 Build Model and Trainable Parameter Phases
## 4.2 Checkpoint Provenance and Head Diagnostics

# 5. Training
## 5.1 Source-Only Classifier
## 5.2 LOSO Target-Fold Runner
## 5.3 Run All Target Subjects

In [ ]:
print("Full CONFIG banner:\n"+json.dumps(CONFIG,indent=2))
paths=locate_subject_files(CONFIG["source_extract_dir"],CONFIG["subjects_to_use"]); all_data={subject_id(p):load_subject(p,CONFIG) for p in paths}; targets={f"sub-{int(s):02d}" for s in CONFIG["target_subjects"]}; missing_targets=targets-set(all_data); assert not missing_targets,f"Target subjects not loaded: {sorted(missing_targets)}"; SUBJECTS=sorted(targets); FOLD_RESULTS=[]; CH_NAMES=next(iter(all_data.values()))[2]; provenance=[]
for target in SUBJECTS:
    source_ids=sorted(set(all_data)-{target}); assert_source_exclusion(target,source_ids); sig=checkpoint_signature(CONFIG,source_ids,CONFIG["model_name"]); ckpt=Path(CONFIG["checkpoint_cache_dir"])/(sig+".pt"); state=None; expected=None
    if CONFIG["adaptation_mode"]!="target_only":
        source_x=np.concatenate([all_data[s][0] for s in source_ids]); source_y=np.concatenate([all_data[s][1] for s in source_ids]); mean,scale=fit_normalizer(source_x,CONFIG["normalization_mode"],CONFIG["normalization_eps"]); normalizer_hash=array_hash(mean,scale); expected={"signature":sig,"target_subject":target,"source_subject_ids":source_ids,"source_shape":list(source_x.shape),"source_class_counts":np.bincount(source_y,minlength=2).tolist(),"source_normalizer_hash":normalizer_hash,"model_name":CONFIG["model_name"],"model_kwargs":CONFIG.get("model_kwargs",{})}
        source_x=((source_x-mean)/scale).astype('float32')
        if ckpt.exists(): state=load_compatible_checkpoint(ckpt,expected)
        else:
            model=build_model(CONFIG["model_name"],source_x.shape[1],source_x.shape[2],CONFIG["target_sfreq"],DEVICE,CONFIG.get("model_kwargs")); configure_adaptation(model,"source_pretrain"); train_eval(model,source_x,source_y,source_x[:8],source_y[:8],CONFIG,BASE_SEED,epochs=CONFIG["source_epochs"],batch_size=CONFIG["source_batch_size"]); save_checkpoint(ckpt,model,expected); state=model.state_dict()
    x,y,_,_=all_data[target]
    for split in make_splits(y,CONFIG,target):
        tr=np.asarray(split["train_indices"]); te=np.asarray(split["test_indices"]); assert not set(tr)&set(te); tmean,tscale=fit_normalizer(x[tr],CONFIG["normalization_mode"],CONFIG["normalization_eps"]); model=build_model(CONFIG["model_name"],x.shape[1],x.shape[2],CONFIG["target_sfreq"],DEVICE,CONFIG.get("model_kwargs"))
        if CONFIG["adaptation_mode"]!="target_only": model.load_state_dict(state)
        head_name,ntrain=configure_adaptation(model,CONFIG["adaptation_mode"]); lr=CONFIG["finetune_learning_rate"] if CONFIG["adaptation_mode"]=="full_finetune" else CONFIG["learning_rate"]
        pred,prob,elapsed=train_eval(model,((x[tr]-tmean)/tscale).astype('float32'),y[tr],((x[te]-tmean)/tscale).astype('float32'),y[te],CONFIG,BASE_SEED,lr=lr,epochs=CONFIG["adaptation_epochs"]); FOLD_RESULTS.append(fold_result(target,split["fold_id"],te,y[te],pred,prob,model,elapsed,BASE_SEED,{"source_subject_ids":source_ids,"checkpoint_signature":sig if state is not None else None,"checkpoint_path":str(ckpt) if state is not None else None,"classifier_head":head_name,"adaptation_mode":CONFIG["adaptation_mode"],"target_train_indices":tr.tolist(),"target_normalizer_hash":array_hash(tmean,tscale)}))
    provenance.append((expected or {"target_subject":target,"source_subject_ids":source_ids,"model_name":CONFIG["model_name"],"source_pretraining_skipped":True})|{"checkpoint_path":str(ckpt) if state is not None else None})
subject_inventory_path=ARTIFACT_DIR/"subject_inventory.csv"; pd.DataFrame([{"subject_id":s,"role":"target" if s in targets else "source"} for s in all_data]).to_csv(subject_inventory_path,index=False); (ARTIFACT_DIR/"checkpoint_provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")

# 6. Results
## 6.1 Aggregate Metrics

In [ ]:
def aggregate_results(rows):
    subject_rows=[]
    for sid in sorted({r["subject_id"] for r in rows}):
        rr=[r for r in rows if r["subject_id"]==sid]; yt=np.concatenate([np.asarray(r["true_labels"]) for r in rr]); yp=np.concatenate([np.asarray(r["predictions"]) for r in rr])
        subject_rows.append({"subject_id":sid,"balanced_accuracy":float(__import__('sklearn').metrics.balanced_accuracy_score(yt,yp)),"accuracy":float((yt==yp).mean()),"n_trials":len(yt)})
    vals=[r["balanced_accuracy"] for r in subject_rows]
    global_metrics={"mean_subject_balanced_accuracy":float(np.mean(vals)),"subject_bootstrap_95_ci":bootstrap_ci(vals,BASE_SEED,CONFIG.get("bootstrap_iterations",10000)),"n_subjects":len(subject_rows),"n_folds_total":len(rows),"collapse_rate":float(np.mean([r["collapse_diagnostics"]["collapsed"] for r in rows])),"mean_training_seconds":float(np.mean([r["training_seconds"] for r in rows]))}
    return subject_rows,global_metrics
SUBJECT_ROWS,GLOBAL_METRICS=aggregate_results(FOLD_RESULTS)

## 6.2 Performance Visualizations

In [ ]:
plt.figure(figsize=(6,3)); plt.bar([r["subject_id"] for r in SUBJECT_ROWS],[100*r["balanced_accuracy"] for r in SUBJECT_ROWS]); plt.axhline(50,color="k",ls="--"); plt.tight_layout(); plt.savefig(ARTIFACT_DIR/"loso_subject_performance.png",dpi=150); plt.close()

## 6.3 Experiment Summary

In [ ]:
print(json.dumps(GLOBAL_METRICS,indent=2))

## 6.4 Adaptation Diagnostics
Checkpoint sources, split indices, classifier head, and trainable counts are persisted per fold.

## 6.5 Save Artifacts

In [ ]:
cv_results_path=ARTIFACT_DIR/"cv_results.json"; cv_results_path.write_text(json.dumps(FOLD_RESULTS,indent=2),encoding="utf-8")
subject_metrics_path=ARTIFACT_DIR/"subject_metrics.json"; subject_metrics_path.write_text(json.dumps(SUBJECT_ROWS,indent=2),encoding="utf-8")
global_metrics_path=ARTIFACT_DIR/"global_metrics.json"; global_metrics_path.write_text(json.dumps(GLOBAL_METRICS,indent=2),encoding="utf-8")
pd.DataFrame(FOLD_RESULTS).to_csv(ARTIFACT_DIR/"fold_results.csv",index=False); pd.DataFrame(SUBJECT_ROWS).to_csv(ARTIFACT_DIR/"subject_results.csv",index=False)
run_metadata={"run_id":RUN_ID,"artifact_dir":str(ARTIFACT_DIR),"experiment_name":CONFIG["experiment_name"],"config_note":CONFIG["config_note"],"subjects":SUBJECTS,"channel_names":CH_NAMES,"model_name":CONFIG.get("model_name"),"seed":BASE_SEED,"global_metrics":GLOBAL_METRICS,"artifacts":{p.name:str(p) for p in ARTIFACT_DIR.iterdir()}}
run_metadata_path=ARTIFACT_DIR/"run_metadata.json"; run_metadata_path.write_text(json.dumps(run_metadata,indent=2),encoding="utf-8")
print(f"CV results saved to:      {cv_results_path}"); print(f"Subject metrics saved to: {subject_metrics_path}"); print(f"Global metrics saved to:  {global_metrics_path}"); print(f"Run metadata saved to:    {run_metadata_path}"); print(f"\nAll artifacts in: {ARTIFACT_DIR}")
try: _LOG_FILE_HANDLE.close()
except Exception: pass